## 1. Setup and Configuration

### 1.1 Environment Setup

#### 1.1.1 Colab Setup

In [ ]:
# 1) Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# 2) Clone or update the repo
BRANCH = "main"  # Change if working on a different branch

# Clone fresh each session 
!rm -rf /content/code-satp
!git clone -b $BRANCH --depth 1 https://github.com/eteitelbaum/code-satp.git /content/code-satp

# 3) Install required packages
!pip install -qU pip setuptools wheel
!pip install --upgrade transformers datasets evaluate rouge-score rapidfuzz

# 4) Make result directories
import pathlib
import sys
pathlib.Path("/content/drive/MyDrive/colab/satp-results").mkdir(parents=True, exist_ok=True)
pathlib.Path("/content/drive/MyDrive/colab/satp-results/location-extraction").mkdir(parents=True, exist_ok=True)

# 5) Import utilities from cloned repo (using importlib to handle hyphens in path)
try:
    import importlib.util
    
    # Import file_io utilities
    spec = importlib.util.spec_from_file_location(
        "file_io",
        "/content/code-satp/models/classification-models/utils/file_io.py"
    )
    file_io = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(file_io)
    
    get_task_results_dir = file_io.get_task_results_dir
    save_dataframe_csv = file_io.save_dataframe_csv
    
    # Import training utilities from count-models
    sys.path.append("/content/code-satp/models/count-models")
    from utils.training_utils import create_seq2seq_training_args
    
    # Define task name for consistent file organization
    TASK_NAME = "location-extraction"
    
    print("="*80)
    print("✅ SETUP COMPLETE")
    print("="*80)
    print(f"📂 Cloned repo: /content/code-satp")
    print(f"📂 Results dir: {get_task_results_dir(TASK_NAME)}")
    print(f"✅ File I/O utilities loaded successfully")
    print("="*80)
    
except Exception as e:
    print("="*80)
    print("⚠️  WARNING: Could not load file_io utilities")
    print(f"Error: {e}")
    print("="*80)
    print("Falling back to local mode - files will be saved to current directory")
    
    # Define fallback functions
    TASK_NAME = "location-extraction"
    
    def get_task_results_dir(task_name):
        return pathlib.Path(f"./results/{task_name}")
    
    def save_dataframe_csv(df, filename, task_name=None):
        output_path = f"./results/{task_name}/{filename}" if task_name else f"./{filename}"
        pathlib.Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(output_path, index=False)
        return output_path
    
    print("="*80)

#### 1.1.2 Local Setup

In [ ]:
# # For local development, uncomment these lines:
# import sys
# import os

# # Local data and results paths
# DATA_PATH = "../../data/"
# RESULTS_PATH = "./results/"

# # Create local results directory
# os.makedirs(RESULTS_PATH, exist_ok=True)

# # Verify GPU availability
# import torch
# if torch.cuda.is_available():
#     print(f"✅ GPU Available: {torch.cuda.get_device_name(0)}")
# else:
#     print("⚠️ GPU not available, using CPU")

# print("✅ Local setup complete!")

### 1.2 Import Libraries

In [ ]:
# Core libraries
import os
import re
import json
import pandas as pd
import numpy as np
from pathlib import Path

# Machine learning and transformers
import torch
from torch.utils.data import Dataset
from transformers import (
    T5Tokenizer, 
    T5ForConditionalGeneration,
    Trainer, 
    TrainingArguments,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from datasets import Dataset as HFDataset

# Evaluation metrics
import evaluate
from sklearn.model_selection import train_test_split

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 2. Data Loading and Preparation

### 2.1 Load Raw Data

In [ ]:
# Try loading from cloned repository first
try:
    data = pd.read_csv('/content/code-satp/data/location_info_augmented.csv', dtype=str)
    print("✅ Data loaded successfully from cloned repository")
    print(f"Rows: {len(data)}, Columns: {len(data.columns)}")
except Exception as e:
    print(f"❌ Error loading CSV from cloned repo: {e}")
    print("🔧 Fallback: Trying GitHub URL...")
    
    # Fallback to GitHub if cloned repo fails or if working locally
    url = "https://raw.githubusercontent.com/eteitelbaum/code-satp/refs/heads/main/data/location_info_augmented.csv"
    try:
        data = pd.read_csv(url, dtype=str)
        print("✅ Data loaded successfully from GitHub")
        print(f"Rows: {len(data)}, Columns: {len(data.columns)}")
    except Exception as e:
        print(f"❌ Error loading CSV from GitHub: {e}")

### 2.2 Create Structured Location Labels

#### About Structured Output Format

This notebook trains a T5-base model to extract location information from incident summaries using a **structured output format** with explicit labels for each administrative level.

**Key Changes from Previous Approach:**

1. **Data Source**
   - Using `location_info_augmented.csv` which contains human-annotated and augmented location data
   - Ground truth comes from separate columns: `state`, `district`, `village_name`, `other_locations`
   - `other_locations` combines filtered blocks (only those in text) + other_areas (forests, police stations, etc.)
   - All duplicates between block/other_areas and village names have been removed

2. **Structured Output Format**
   - **Old Format:** `"state: Chhattisgarh, district: Sukma, block: Dornapal, village: Nilamadgu"` (block often missing)
   - **New Format:** `"state: Chhattisgarh, district: Sukma, village: Nilamadgu, other_locations: Dornapal"` (more robust)
   
   **Benefits:**
   - Clear hierarchy with labeled levels
   - `other_locations` provides flexible context (blocks, forests, police stations, etc.)
   - Better extractability: all values in `other_locations` appear in incident text
   - Expected F1 improvement: 70-75% (vs 41% for blocks)
   - Handles missing levels gracefully
   - Enables level-specific evaluation metrics

3. **Enhanced Evaluation Metrics**
   Instead of simple token matching, we now compute:
   - **Exact Match Accuracy**: All levels must match perfectly
   - **Per-Level Metrics**: Precision, Recall, F1 for each of state/district/village/other_locations
   - **Micro-averaged F1**: Overall performance across all levels
   - **Level-by-level breakdown**: See exactly which administrative levels the model struggles with

4. **Better for Downstream Tasks**
   The structured output can be directly:
   - Parsed into a dictionary for programmatic access
   - Fed into geocoding APIs with component filtering (state/district as filters, village+other_locations as address)
   - Used for geographic analysis at specific administrative levels
   - Validated against known location hierarchies

#### Build Structured Location Column

In [ ]:
# Create structured human_annotated_location column from individual location fields
def build_structured_location(row):
    """Build structured location string from state, district, village, other_locations columns."""
    parts = []
    
    if pd.notna(row.get('state')) and str(row.get('state')).strip():
        parts.append(f"state: {str(row['state']).strip()}")
    
    if pd.notna(row.get('district')) and str(row.get('district')).strip():
        parts.append(f"district: {str(row['district']).strip()}")
    
    if pd.notna(row.get('village_name')) and str(row.get('village_name')).strip():
        parts.append(f"village: {str(row['village_name']).strip()}")
    
    if pd.notna(row.get('other_locations')) and str(row.get('other_locations')).strip():
        parts.append(f"other_locations: {str(row['other_locations']).strip()}")
    
    return ', '.join(parts) if parts else ''

# Apply to create the human_annotated_location column
data['human_annotated_location'] = data.apply(build_structured_location, axis=1)

# Display sample to verify
print("Sample human-annotated locations:")
display(data[['state', 'district', 'village_name', 'other_locations', 'human_annotated_location']].head(10))

#### Verify Data Structure

In [ ]:
# Display the first few rows to verify the data structure
print("Dataset shape:", data.shape)
print("\nFirst few rows:")
display(data.head())

### 2.3 Data Preprocessing and Filtering

#### Create Train/Validation/Test Splits

In [ ]:
from datasets import Dataset, DatasetDict

def prepare_data(df):
    # Updated prompt with structured output format
    inputs = [
        f"Extract location hierarchy from incident: {summary}\nFormat: state: <name>, district: <name>, village: <name>, other_locations: <name>. Use exact format with labels. Omit missing levels."
        for summary in df['incident_summary']
    ]
    targets = df['human_annotated_location'].tolist()
    
    # Preserve metadata columns (date, incident_number, incident_summary) for later use in evaluation/geocoding
    # Convert datetime to string format to avoid PyArrow NaT issues
    dates = []
    for d in df['date']:
        if pd.isna(d):
            dates.append(None)
        else:
            # Convert to string format YYYY-MM-DD
            try:
                dates.append(pd.to_datetime(d).strftime('%Y-%m-%d'))
            except:
                dates.append(None)
    
    # Preserve incident identifiers for matching with baseline coordinates later
    # Convert to string explicitly to avoid PyArrow type inference issues
    incident_numbers = [str(x) if pd.notna(x) else None for x in df['incident_number']]
    incident_summaries = [str(x) if pd.notna(x) else None for x in df['incident_summary']]
    
    return {
        'input_text': inputs, 
        'target_text': targets, 
        'date': dates,
        'incident_number': incident_numbers,
        'incident_summary': incident_summaries
    }

# TEMPORAL SPLIT: Sort by date for chronological train/val/test split
# This ensures clean separation of Telangana formation (June 2, 2014)
# Training on earlier data, testing on more recent data (realistic deployment scenario)
data['date'] = pd.to_datetime(data['date'], errors='coerce')
data = data.sort_values('date').reset_index(drop=True)

# Calculate split indices for 80/10/10 split
train_end_idx = int(len(data) * 0.8)
val_end_idx = int(len(data) * 0.9)

# Create temporal splits
train_data = data.iloc[:train_end_idx]
val_data = data.iloc[train_end_idx:val_end_idx]
test_data = data.iloc[val_end_idx:]

print(f"Temporal split summary:")
print(f"  Training:   {len(train_data)} incidents ({train_data['date'].min()} to {train_data['date'].max()})")
print(f"  Validation: {len(val_data)} incidents ({val_data['date'].min()} to {val_data['date'].max()})")
print(f"  Test:       {len(test_data)} incidents ({test_data['date'].min()} to {test_data['date'].max()})")
print(f"\nNote: Training ends before Telangana formation (June 2, 2014)")
print(f"      Test set contains only post-2014 incidents with current state names")

# Create datasets from temporal splits
dataset = DatasetDict({
    'train': Dataset.from_dict(prepare_data(train_data)),
    'validation': Dataset.from_dict(prepare_data(val_data)),
    'test': Dataset.from_dict(prepare_data(test_data))
})

print(f"\n{dataset}")


DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 7854
    })
    validation: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 982
    })
    test: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 982
    })
})


## 3. Model Setup and Training

**Padding Token Workflow:**
1. **During preprocessing** (Section 3.1): Padding tokens in labels are replaced with `-100`
2. **During training**: The loss function automatically ignores tokens marked as `-100`
3. **During decoding** (Section 5+): Filter out `-100` tokens before decoding: `[l for l in label if l != -100]`

**Optimization:**
- Training optimizes **cross-entropy loss** (`eval_loss`) on validation set
- Task-specific metrics (exact match, per-level F1) are computed **after training** for evaluation
- BLEU/ROUGE metrics are not used (better suited for text generation than structured extraction)

### 3.1 Initialize Tokenizer and Preprocessing

In [ ]:
from transformers import T5Tokenizer

# Initialize the tokenizer
model_name = "t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)

# Tokenization function
def preprocess_function(examples):
    inputs = examples['input_text']
    targets = examples['target_text']

    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding='max_length'
    )

    # Tokenize targets and get input_ids
    labels = tokenizer(
        targets,
        max_length=64,
        truncation=True,
        padding='max_length'
    ).input_ids

    # Replace padding token id's in the labels with -100 so they are ignored by the loss
    labels = [[(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels]

    model_inputs['labels'] = labels
    return model_inputs

# Apply tokenization to all splits
tokenized_datasets = dataset.map(preprocess_function, batched=True)

# Set format for PyTorch
tokenized_datasets.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'labels']
)

print("Tokenization complete!")
print(f"Train set: {len(tokenized_datasets['train'])} examples")
print(f"Validation set: {len(tokenized_datasets['validation'])} examples")
print(f"Test set: {len(tokenized_datasets['test'])} examples")


### 3.2 GPU Configuration

In [ ]:
import torch
import numpy as np

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f'Random seed set to: {SEED}')

### 3.4 Initialize Model and Training Configuration

**Training Strategy:**
- **10 epochs maximum** with **early stopping** (patience=3 epochs)
- Training stops if validation loss doesn't improve for 3 consecutive epochs
- Prevents overfitting while allowing model to fully converge
- Saves only the 3 best checkpoints to conserve disk space
- Final model is the one with lowest validation loss

In [ ]:
from transformers import T5ForConditionalGeneration, Seq2SeqTrainer, EarlyStoppingCallback
import evaluate

# Initialize the model
model = T5ForConditionalGeneration.from_pretrained(model_name)
model.to(device)

# Training configuration constants
BATCH_SIZE = 8
MAX_EPOCHS = 10
LEARNING_RATE = 3e-5
GENERATION_MAX_LENGTH = 40  

# Task-specific parameters (batch_size, learning_rate, epochs, generation_max_length) are set here.
# Standard settings (optimizer, mixed precision, save_total_limit, etc.) are configured in 
# models/count-models/utils/training_utils.py - see create_seq2seq_training_args() for details.
training_args = create_seq2seq_training_args(
    output_dir="results/model_checkpoints/t5-base",
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    num_epochs=MAX_EPOCHS,
    seed=SEED,
    generation_max_length=GENERATION_MAX_LENGTH
)

# Override location-extraction-specific settings

# Initialize the Trainer with early stopping callback
# Early stopping: stops training if eval_loss doesn't improve for 3 epochs
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001)],
    # compute_metrics=compute_metrics
)

### 3.5 Train the Model

In [ ]:

# Train the model
trainer.train()


Epoch,Training Loss,Validation Loss
1,0.089200,0.073098
2,0.074200,0.066961
3,0.071400,0.064558


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=2946, training_loss=0.13490951619662966, metrics={'train_runtime': 3414.9844, 'train_samples_per_second': 6.9, 'train_steps_per_second': 0.863, 'total_flos': 1.434826581737472e+16, 'train_loss': 0.13490951619662966, 'epoch': 3.0})

### 3.6 Save Trained Model

In [ ]:
# Save the model and tokenizer to standardized location
try:
    results_dir = get_task_results_dir(TASK_NAME)
    model_dir = results_dir / "t5base_finetuned_model"
    model.save_pretrained(model_dir)
    tokenizer.save_pretrained(model_dir)
    print(f"✅ Model and tokenizer saved to: {model_dir}")
except NameError:
    # Fallback to Drive path for Colab
    model_path = '/content/drive/MyDrive/colab/satp-results/location-extraction/t5base_finetuned_model'
    os.makedirs(model_path, exist_ok=True)
    model.save_pretrained(model_path)
    tokenizer.save_pretrained(model_path)
    print(f"✅ Model and tokenizer saved to: {model_path}")

('/content/drive/MyDrive/SATP_data/location_context_extraction_exp/t5base_finetuned_model/tokenizer_config.json',
 '/content/drive/MyDrive/SATP_data/location_context_extraction_exp/t5base_finetuned_model/special_tokens_map.json',
 '/content/drive/MyDrive/SATP_data/location_context_extraction_exp/t5base_finetuned_model/spiece.model',
 '/content/drive/MyDrive/SATP_data/location_context_extraction_exp/t5base_finetuned_model/added_tokens.json')

## 4. Evaluation Metrics Definition

This section defines the evaluation functions used to assess model performance on the temporal test set. These functions are used in Section 5 after loading the trained model.

### 4.1 Parse Structured Location Output

In [ ]:
def parse_structured_location(text):
    """Parse structured location string into a dictionary."""
    location_dict = {
        'state': None,
        'district': None,
        'village': None,
        'other_locations': None
    }
    
    if not text or text.strip() == '':
        return location_dict
    
    # Split by comma and parse each part
    parts = [part.strip() for part in text.split(',')]
    for part in parts:
        if ':' in part:
            label, value = part.split(':', 1)
            label = label.strip().lower()
            value = value.strip()
            if label in location_dict:
                location_dict[label] = value
    
    return location_dict

# Test the parser with a sample
sample_location = "state: Chhattisgarh, district: Sukma, village: Nilamadgu, other_locations: Dornapal"
parsed = parse_structured_location(sample_location)
print("Sample parsing test:")
print(f"Input: {sample_location}")
print(f"Parsed: {parsed}")

### 4.2 Dual Metrics: Exact vs Fuzzy Matching

We compute **two sets of metrics** to comprehensively evaluate location extraction:

1. **Exact Match Metrics** (strict)
   - Requires perfect string match (case-insensitive)
   - Example: `"Vishakhapatnam"` ≠ `"Visakhapatnam"` ❌
   - Best for: Understanding model's exact accuracy

2. **Fuzzy Match Metrics** (lenient, 85% similarity threshold)
   - Handles spelling variations using Levenshtein distance
   - Example: `"Vishakhapatnam"` ≈ `"Visakhapatnam"` ✅ (similarity: 93%)
   - Best for: Real-world usability where spelling variations are common

**Special Handling for `other_locations`:**
- Uses **token-level F1** (since it's multi-value: `"Dornapal, Chintagufa Forest"`)
- Order-independent: `"A, B"` = `"B, A"`
- Partial credit: Extracting 1 of 2 locations gives 50% recall
- Both exact and fuzzy matching applied at token level

**Why Both Metrics Matter:**
- **Exact F1**: Shows if model needs more training data or better architecture
- **Fuzzy F1**: Shows real-world performance for geocoding (where small variations don't matter)
- **Gap between them**: Indicates how much performance loss is due to spelling variations vs actual errors

In [ ]:
from rapidfuzz import fuzz

def fuzzy_match(str1, str2, threshold=85):
    """
    Check if two strings match with fuzzy matching (handles spelling variations).
    
    Returns True if:
    - Both strings are None (both fields are empty)
    - Both strings exist and have >= threshold similarity
    
    Returns False if:
    - Only one string is None (one field exists, the other doesn't)
    - Both strings exist but similarity < threshold
    """
    # Both None means both fields are empty - this is a match
    if str1 is None and str2 is None:
        return True
    # One is None but not the other - this is NOT a match
    if str1 is None or str2 is None:
        return False
    # Both exist - check fuzzy similarity
    return fuzz.ratio(str1.lower(), str2.lower()) >= threshold

def compute_detailed_metrics(predictions, labels):
    """
    Compute detailed metrics with both exact and fuzzy matching.
    
    Returns metrics at three levels:
    1. Full record accuracy: % of examples where ALL fields match
    2. Micro-averaged metrics: Aggregate P/R/F1 across all field extractions
       - Calculated by summing correct/predicted/actual across all fields
       - Gives equal weight to each field extraction (not each example)
    3. Field-level metrics: Individual P/R/F1 for state, district, village, other_locations
    
    Note: Fuzzy match accuracy (full record) is typically low because it requires
    ALL fields to fuzzy match. Field-level fuzzy metrics are more informative.
    """
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = [
        tokenizer.decode([l for l in label if l != -100], skip_special_tokens=True)
        for label in labels
    ]
    
    # Initialize counters for EXACT matching
    exact_matches = 0
    exact_core_matches = 0  # state + district + village only
    exact_level_metrics = {
        'state': {'correct': 0, 'predicted': 0, 'total': 0},
        'district': {'correct': 0, 'predicted': 0, 'total': 0},
        'village': {'correct': 0, 'predicted': 0, 'total': 0},
        'other_locations': {'correct': 0, 'predicted': 0, 'total': 0}
    }
    
    # Initialize counters for FUZZY matching
    fuzzy_matches = 0
    fuzzy_core_matches = 0  # state + district + village only
    fuzzy_level_metrics = {
        'state': {'correct': 0, 'predicted': 0, 'total': 0},
        'district': {'correct': 0, 'predicted': 0, 'total': 0},
        'village': {'correct': 0, 'predicted': 0, 'total': 0},
        'other_locations': {'correct': 0, 'predicted': 0, 'total': 0}
    }
    
    total_examples = len(decoded_preds)
    
    for pred, label in zip(decoded_preds, decoded_labels):
        pred_dict = parse_structured_location(pred)
        label_dict = parse_structured_location(label)
        
        # Check exact match for entire prediction (all 4 fields)
        if pred_dict == label_dict:
            exact_matches += 1
        
        # Check exact match for core geographic hierarchy (state + district + village only)
        core_exact_match = True
        for level in ['state', 'district', 'village']:
            if pred_dict[level] != label_dict[level]:
                core_exact_match = False
                break
        if core_exact_match:
            exact_core_matches += 1
        
        # Check fuzzy match for entire prediction (all 4 fields must fuzzy match)
        all_levels_fuzzy_match = True
        for level in ['state', 'district', 'village', 'other_locations']:
            if not fuzzy_match(pred_dict[level], label_dict[level]):
                all_levels_fuzzy_match = False
                break
        if all_levels_fuzzy_match:
            fuzzy_matches += 1
        
        # Check fuzzy match for core geographic hierarchy (state + district + village only)
        core_fuzzy_match = True
        for level in ['state', 'district', 'village']:
            if not fuzzy_match(pred_dict[level], label_dict[level]):
                core_fuzzy_match = False
                break
        if core_fuzzy_match:
            fuzzy_core_matches += 1
        
        # Compute per-level metrics (EXACT)
        for level in ['state', 'district', 'village']:
            if label_dict[level] is not None:
                exact_level_metrics[level]['total'] += 1
            if pred_dict[level] is not None:
                exact_level_metrics[level]['predicted'] += 1
            if pred_dict[level] is not None and label_dict[level] is not None:
                if pred_dict[level].lower() == label_dict[level].lower():
                    exact_level_metrics[level]['correct'] += 1
        
        # Compute per-level metrics (FUZZY)
        for level in ['state', 'district', 'village']:
            if label_dict[level] is not None:
                fuzzy_level_metrics[level]['total'] += 1
            if pred_dict[level] is not None:
                fuzzy_level_metrics[level]['predicted'] += 1
            if pred_dict[level] is not None and label_dict[level] is not None:
                if fuzzy_match(pred_dict[level], label_dict[level]):
                    fuzzy_level_metrics[level]['correct'] += 1
        
        # Special handling for other_locations (token-level F1) - EXACT
        if label_dict['other_locations'] is not None:
            label_tokens = set([t.strip().lower() for t in label_dict['other_locations'].split(',') if t.strip()])
            exact_level_metrics['other_locations']['total'] += len(label_tokens)
            
            if pred_dict['other_locations'] is not None:
                pred_tokens = set([t.strip().lower() for t in pred_dict['other_locations'].split(',') if t.strip()])
                exact_level_metrics['other_locations']['predicted'] += len(pred_tokens)
                exact_level_metrics['other_locations']['correct'] += len(pred_tokens & label_tokens)
            # If prediction is None but label exists, predicted stays 0
        elif pred_dict['other_locations'] is not None:
            # Prediction exists but label is None (false positives)
            pred_tokens = set([t.strip().lower() for t in pred_dict['other_locations'].split(',') if t.strip()])
            exact_level_metrics['other_locations']['predicted'] += len(pred_tokens)
        
        # Special handling for other_locations (token-level F1) - FUZZY
        if label_dict['other_locations'] is not None:
            label_tokens = [t.strip() for t in label_dict['other_locations'].split(',') if t.strip()]
            fuzzy_level_metrics['other_locations']['total'] += len(label_tokens)
            
            if pred_dict['other_locations'] is not None:
                pred_tokens = [t.strip() for t in pred_dict['other_locations'].split(',') if t.strip()]
                fuzzy_level_metrics['other_locations']['predicted'] += len(pred_tokens)
                
                # For fuzzy matching, count matches using fuzzy string similarity
                matched_pred = set()
                matched_label = set()
                for p_idx, p_token in enumerate(pred_tokens):
                    for l_idx, l_token in enumerate(label_tokens):
                        if l_idx not in matched_label and fuzzy_match(p_token, l_token):
                            fuzzy_level_metrics['other_locations']['correct'] += 1
                            matched_pred.add(p_idx)
                            matched_label.add(l_idx)
                            break
        elif pred_dict['other_locations'] is not None:
            # Prediction exists but label is None (false positives)
            pred_tokens = [t.strip() for t in pred_dict['other_locations'].split(',') if t.strip()]
            fuzzy_level_metrics['other_locations']['predicted'] += len(pred_tokens)
    
    # Compute precision, recall, F1 for each level (EXACT)
    results = {
        'exact_match': exact_matches / total_examples * 100,
        'exact_core_match': exact_core_matches / total_examples * 100,
        'total_examples': total_examples
    }
    
    for level in ['state', 'district', 'village', 'other_locations']:
        metrics = exact_level_metrics[level]
        precision = (metrics['correct'] / metrics['predicted'] * 100) if metrics['predicted'] > 0 else 0
        recall = (metrics['correct'] / metrics['total'] * 100) if metrics['total'] > 0 else 0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0
        
        results[f'{level}_exact_precision'] = precision
        results[f'{level}_exact_recall'] = recall
        results[f'{level}_exact_f1'] = f1
        results[f'{level}_support'] = metrics['total']
    
    # Compute precision, recall, F1 for each level (FUZZY)
    results['fuzzy_match'] = fuzzy_matches / total_examples * 100
    results['fuzzy_core_match'] = fuzzy_core_matches / total_examples * 100
    
    for level in ['state', 'district', 'village', 'other_locations']:
        metrics = fuzzy_level_metrics[level]
        precision = (metrics['correct'] / metrics['predicted'] * 100) if metrics['predicted'] > 0 else 0
        recall = (metrics['correct'] / metrics['total'] * 100) if metrics['total'] > 0 else 0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0
        
        results[f'{level}_fuzzy_precision'] = precision
        results[f'{level}_fuzzy_recall'] = recall
        results[f'{level}_fuzzy_f1'] = f1
    
    # Compute micro-averaged F1 (weighted by support) - EXACT
    total_correct = sum(exact_level_metrics[level]['correct'] for level in ['state', 'district', 'village', 'other_locations'])
    total_predicted = sum(exact_level_metrics[level]['predicted'] for level in ['state', 'district', 'village', 'other_locations'])
    total_actual = sum(exact_level_metrics[level]['total'] for level in ['state', 'district', 'village', 'other_locations'])
    
    micro_exact_precision = (total_correct / total_predicted * 100) if total_predicted > 0 else 0
    micro_exact_recall = (total_correct / total_actual * 100) if total_actual > 0 else 0
    micro_exact_f1 = (2 * micro_exact_precision * micro_exact_recall / (micro_exact_precision + micro_exact_recall)) if (micro_exact_precision + micro_exact_recall) > 0 else 0
    
    results['micro_exact_precision'] = micro_exact_precision
    results['micro_exact_recall'] = micro_exact_recall
    results['micro_exact_f1'] = micro_exact_f1
    
    # Compute micro-averaged F1 (weighted by support) - FUZZY
    total_correct = sum(fuzzy_level_metrics[level]['correct'] for level in ['state', 'district', 'village', 'other_locations'])
    total_predicted = sum(fuzzy_level_metrics[level]['predicted'] for level in ['state', 'district', 'village', 'other_locations'])
    total_actual = sum(fuzzy_level_metrics[level]['total'] for level in ['state', 'district', 'village', 'other_locations'])
    
    micro_fuzzy_precision = (total_correct / total_predicted * 100) if total_predicted > 0 else 0
    micro_fuzzy_recall = (total_correct / total_actual * 100) if total_actual > 0 else 0
    micro_fuzzy_f1 = (2 * micro_fuzzy_precision * micro_fuzzy_recall / (micro_fuzzy_precision + micro_fuzzy_recall)) if (micro_fuzzy_precision + micro_fuzzy_recall) > 0 else 0
    
    results['micro_fuzzy_precision'] = micro_fuzzy_precision
    results['micro_fuzzy_recall'] = micro_fuzzy_recall
    results['micro_fuzzy_f1'] = micro_fuzzy_f1
    
    # Print summary - formatted as a table
    print("="*80)
    print("EVALUATION RESULTS")
    print("="*80)
    print(f"Total Examples: {total_examples}")
    print()
    
    # Overall accuracy (full record matches)
    print(f"{'FULL RECORD ACCURACY':<40} {'Exact Match':<15} {'Fuzzy Match':<15}")
    print("-" * 70)
    print(f"{'All fields correct (incl. other_locations)':<40} {results['exact_match']:>6.2f}%        {results['fuzzy_match']:>6.2f}%")
    print(f"{'Core geography correct (state+district+village)':<40} {results['exact_core_match']:>6.2f}%        {results['fuzzy_core_match']:>6.2f}%")
    print()
    print("Note: 'Core geography' excludes other_locations field to focus on the main")
    print("      administrative hierarchy (state → district → village).")
    print()
    
    # Micro-averaged metrics (aggregated across all fields)
    print(f"{'MICRO-AVERAGED METRICS':<40} {'Exact Match':<15} {'Fuzzy Match':<15}")
    print("(Aggregated across all fields: state, district, village, other_locations)")
    print("-" * 70)
    print(f"{'Precision':<40} {micro_exact_precision:>6.2f}%        {micro_fuzzy_precision:>6.2f}%")
    print(f"{'Recall':<40} {micro_exact_recall:>6.2f}%        {micro_fuzzy_recall:>6.2f}%")
    print(f"{'F1 Score':<40} {micro_exact_f1:>6.2f}%        {micro_fuzzy_f1:>6.2f}%")
    print()
    
    # Field-level metrics table
    print(f"{'FIELD-LEVEL METRICS':<20} {'Match Type':<12} {'Precision':<12} {'Recall':<12} {'F1 Score':<12} {'Support':<10}")
    print("-" * 78)
    
    for level in ['state', 'district', 'village', 'other_locations']:
        level_display = level.replace('_', ' ').title()
        
        # Exact match row
        print(f"{level_display:<20} {'Exact':<12} "
              f"{results[f'{level}_exact_precision']:>6.2f}%     "
              f"{results[f'{level}_exact_recall']:>6.2f}%     "
              f"{results[f'{level}_exact_f1']:>6.2f}%     "
              f"{results[f'{level}_support']:>6.0f}")
        
        # Fuzzy match row
        print(f"{'':<20} {'Fuzzy':<12} "
              f"{results[f'{level}_fuzzy_precision']:>6.2f}%     "
              f"{results[f'{level}_fuzzy_recall']:>6.2f}%     "
              f"{results[f'{level}_fuzzy_f1']:>6.2f}%     "
              f"{'':<10}")
        print()
    
    print("="*80)
    print("\nKEY INSIGHTS:")
    print("- Micro-averaged metrics: Aggregate performance across all field extractions")
    print("- Field-level metrics: Individual performance for each location component")
    print("- Support: Number of examples in test set with this field present")
    print("- Fuzzy matching: Allows for minor spelling variations (85% similarity threshold)")
    print("="*80)
    
    return results

## 5. Model Evaluation on Test Set

**Workflow:**
- Load a previously trained model (from Section 3 or a saved checkpoint)
- Evaluate it on the temporal test set (from Section 2) using metrics defined in Section 4
- Generate predictions and save detailed results

**Important:** This section requires:
- Section 2 to be run first (to create the temporal split)
- Section 4 to be run first (to define evaluation functions)
- Either Section 3 (to train a new model) or a pre-existing saved model

### 5.1 Load Model and Tokenizer

In [ ]:
import torch
# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

from transformers import T5ForConditionalGeneration, T5Tokenizer

# Load the model and tokenizer from standardized location
try:
    results_dir = get_task_results_dir(TASK_NAME)
    model_path = results_dir / "t5base_finetuned_model"
    print(f"Loading model from: {model_path}")
except NameError:
    # Fallback to Drive path
    model_path = '/content/drive/MyDrive/colab/satp-results/location-extraction/t5base_finetuned_model'
    print(f"Loading model from: {model_path}")

model = T5ForConditionalGeneration.from_pretrained(model_path)
tokenizer = T5Tokenizer.from_pretrained(model_path)
model.to(device)
print(f'✅ Model loaded successfully')
print(f'Using device: {device}')

Using device: cuda
Using device: cuda


### 5.2 Prepare Test Dataset for Inference

**Note:** This section uses the temporal test dataset created in Section 2.3. Make sure to run cells in order (from Section 2 onwards) to ensure the `dataset` variable with the temporal split is available.

In [ ]:
import pandas as pd

# Use the temporal test dataset that was created in Section 2.3
# Note: Requires running cells in order (Section 2.3 must be run first)
print(f"Using temporal test dataset created in Section 2.3")
print(f"Dataset size: {len(dataset['test'])} incidents")
print(f"Features: {', '.join(dataset['test'].features.keys())}")

# Display sample
print("\nSample from test set:")
for i in range(min(3, len(dataset['test']))):
    print(f"\nExample {i+1}:")
    print(f"  Input:  {dataset['test'][i]['input_text'][:100]}...")
    print(f"  Target: {dataset['test'][i]['target_text']}")
    if 'date' in dataset['test'][i]:
        print(f"  Date:   {dataset['test'][i]['date']}")

def preprocess_function(examples):
    inputs = examples['input_text']
    targets = examples['target_text']

    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding='max_length')

    # Tokenize targets with the `text_target` keyword argument
    labels = tokenizer(targets, max_length=64, truncation=True, padding='max_length')

    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Apply tokenization to the temporal test set
tokenized_dataset_test = dataset['test'].map(preprocess_function, batched=True)

# Set format for PyTorch
tokenized_dataset_test.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'labels']
)

print(f"\n✅ Test dataset tokenized and ready for inference")


Map:   0%|          | 0/982 [00:00<?, ? examples/s]

In [ ]:
import torch
from torch.utils.data import DataLoader

def evaluate_model(model, dataloader, tokenizer):
    model.eval()  # Set the model to evaluation mode
    all_predictions = []
    all_labels = []

    with torch.no_grad():  # Disable gradient computation
        for batch in dataloader:
            # Move input data to the appropriate device
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"]

            # Generate predictions
            outputs = model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_length=64
            )
            predictions = outputs.cpu().numpy()  # Move predictions to CPU
            all_predictions.extend(predictions)
            all_labels.extend(labels.numpy())  # Move labels to CPU

    # Compute detailed metrics using utility function
    metrics = compute_metrics(all_predictions, all_labels, tokenizer)
    return metrics

### 5.3 Run Evaluation on Test Set

Uses the evaluation functions defined in Section 4 to compute comprehensive metrics on the temporal test set.

**Understanding the Evaluation Metrics:**

1. **Full Record Accuracy**: Percentage of examples where fields match
   - **All fields correct**: All 4 fields (state, district, village, other_locations) must match
   - **Core geography correct**: Only state + district + village must match (excludes other_locations)
   - Exact Match: Character-perfect matching
   - Fuzzy Match: 85% similarity threshold (allows minor spelling variations)

2. **Micro-Averaged Metrics**: Aggregated performance across all field extractions
   - Sums correct/predicted/actual counts across all 4 fields
   - Gives equal weight to each field extraction (not each example)
   - Better indicator of overall extraction quality than full record accuracy

3. **Field-Level Metrics**: Individual Precision/Recall/F1 for each location component
   - State, District, Village: Direct string matching
   - Other Locations: Token-level matching (handles comma-separated lists)
   - Most granular view of model performance

In [ ]:
from torch.utils.data import DataLoader

# Convert the test dataset to a PyTorch DataLoader
test_loader = DataLoader(tokenized_dataset_test, batch_size=8)


In [ ]:
# Evaluate the model on the test set
metrics = evaluate_model(model, test_loader, tokenizer)

# Print formatted metrics
print_metrics(metrics, model_name="T5-base")

# Also show the raw structure for reference
print("\n" + "="*80)
print("METRICS STRUCTURE (for programmatic access):")
print("="*80)
print(f"Keys: {list(metrics.keys())}")
print(f"Overall metrics: {list(metrics['overall'].keys())}")
print(f"Level metrics: {list(metrics['levels'].keys())}")


Test Set Metrics: {'exact_match': 40.52953156822811}


### 5.4 Save Evaluation Metrics

In [ ]:
# Flatten metrics for CSV export using utility function
flat_metrics = flatten_metrics_for_csv(metrics)

# Create DataFrame with flattened metrics
metrics_df = pd.DataFrame([flat_metrics])
metrics_df['model'] = 'T5-base'
metrics_df['dataset'] = 'test'
metrics_df['task'] = 'location-extraction'

# Reorder columns for clarity - both exact and fuzzy metrics
cols_order = [
    'task', 'model', 'dataset', 
    'exact_match', 'exact_core_match', 'fuzzy_match', 'fuzzy_core_match',
    'micro_exact_f1', 'micro_exact_precision', 'micro_exact_recall',
    'micro_fuzzy_f1', 'micro_fuzzy_precision', 'micro_fuzzy_recall',
    'state_exact_f1', 'state_exact_precision', 'state_exact_recall',
    'state_fuzzy_f1', 'state_fuzzy_precision', 'state_fuzzy_recall',
    'district_exact_f1', 'district_exact_precision', 'district_exact_recall',
    'district_fuzzy_f1', 'district_fuzzy_precision', 'district_fuzzy_recall',
    'village_exact_f1', 'village_exact_precision', 'village_exact_recall',
    'village_fuzzy_f1', 'village_fuzzy_precision', 'village_fuzzy_recall',
    'other_locations_exact_f1', 'other_locations_exact_precision', 'other_locations_exact_recall',
    'other_locations_fuzzy_f1', 'other_locations_fuzzy_precision', 'other_locations_fuzzy_recall',
    'state_support', 'district_support', 'village_support', 'other_locations_support',
    'total_examples'
]
metrics_df = metrics_df[[col for col in cols_order if col in metrics_df.columns]]

# Save using the standard utility function
try:
    saved_path = save_dataframe_csv(metrics_df, "location_extraction_test_metrics.csv", task_name=TASK_NAME)
    print(f"✅ Evaluation metrics saved to: {saved_path}")
except NameError:
    # Fallback if file_io utils not available
    output_path = "./location_extraction_test_metrics.csv"
    metrics_df.to_csv(output_path, index=False)
    print(f"✅ Evaluation metrics saved to: {output_path}")

display(metrics_df)

### 5.5 Generate and Save Test Predictions

In [ ]:
# Generate and save predictions for the entire test set
print("Generating predictions for full test set...")

model.eval()
all_predictions = []
all_labels = []
all_inputs = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]
        
        # Generate predictions
        outputs = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_length=64
        )
        
        # Decode predictions and inputs
        preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        inputs = tokenizer.batch_decode(input_ids, skip_special_tokens=True)
        
        # Decode labels: filter out -100 (padding tokens ignored during training)
        truths = [
            tokenizer.decode([l for l in label if l != -100], skip_special_tokens=True)
            for label in labels
        ]
        
        all_predictions.extend(preds)
        all_labels.extend(truths)
        all_inputs.extend(inputs)

# Create predictions DataFrame
predictions_df = pd.DataFrame({
    'input_text': all_inputs,
    'ground_truth': all_labels,
    'prediction': all_predictions
})

# Add metadata columns from test dataset for geocoding and analysis
predictions_df['date'] = dataset['test']['date']
predictions_df['incident_number'] = dataset['test']['incident_number']
predictions_df['incident_summary'] = dataset['test']['incident_summary']

# Parse structured outputs
predictions_df['gt_state'] = predictions_df['ground_truth'].apply(lambda x: parse_structured_location(x).get('state'))
predictions_df['gt_district'] = predictions_df['ground_truth'].apply(lambda x: parse_structured_location(x).get('district'))
predictions_df['gt_other_locations'] = predictions_df['ground_truth'].apply(lambda x: parse_structured_location(x).get('other_locations'))
predictions_df['gt_village'] = predictions_df['ground_truth'].apply(lambda x: parse_structured_location(x).get('village'))

predictions_df['pred_state'] = predictions_df['prediction'].apply(lambda x: parse_structured_location(x).get('state'))
predictions_df['pred_district'] = predictions_df['prediction'].apply(lambda x: parse_structured_location(x).get('district'))
predictions_df['pred_other_locations'] = predictions_df['prediction'].apply(lambda x: parse_structured_location(x).get('other_locations'))
predictions_df['pred_village'] = predictions_df['prediction'].apply(lambda x: parse_structured_location(x).get('village'))

# Add match indicators
predictions_df['exact_match'] = predictions_df.apply(
    lambda row: parse_structured_location(row['prediction']) == parse_structured_location(row['ground_truth']), 
    axis=1
)
predictions_df['state_match'] = predictions_df.apply(
    lambda row: (row['pred_state'] or '').lower() == (row['gt_state'] or '').lower() if row['gt_state'] else pd.NA,
    axis=1
)
predictions_df['district_match'] = predictions_df.apply(
    lambda row: (row['pred_district'] or '').lower() == (row['gt_district'] or '').lower() if row['gt_district'] else pd.NA,
    axis=1
)
predictions_df['other_locations_match'] = predictions_df.apply(
    lambda row: (row['pred_other_locations'] or '').lower() == (row['gt_other_locations'] or '').lower() if row['gt_other_locations'] else pd.NA,
    axis=1
)
predictions_df['village_match'] = predictions_df.apply(
    lambda row: (row['pred_village'] or '').lower() == (row['gt_village'] or '').lower() if row['gt_village'] else pd.NA,
    axis=1
)

# Save predictions
try:
    saved_path = save_dataframe_csv(predictions_df, "location_extraction_test_predictions.csv", task_name=TASK_NAME)
    print(f"✅ Test predictions saved to: {saved_path}")
    print(f"   Total predictions: {len(predictions_df)}")
    print(f"   Exact matches: {predictions_df['exact_match'].sum()} ({predictions_df['exact_match'].sum()/len(predictions_df)*100:.1f}%)")
except NameError:
    output_path = "./location_extraction_test_predictions.csv"
    predictions_df.to_csv(output_path, index=False)
    print(f"✅ Test predictions saved to: {output_path}")

display(predictions_df.head(10))

### 5.6 Verify Saved Results

In [ ]:
# List all files in the results directory
try:
    results_dir = get_task_results_dir(TASK_NAME)
    print(f"Results directory: {results_dir}\n")
    print("Saved files:")
    print("\n".join(sorted(os.listdir(results_dir))))
except NameError:
    print("Results saved in current directory:")
    import glob
    csv_files = glob.glob("location_extraction*.csv")
    if csv_files:
        print("\n".join(sorted(csv_files)))
    else:
        print("No result files found yet.")

In [ ]:
# Quick view: Show 10 random examples with side-by-side comparison
model.eval()
batch = next(iter(test_loader))

input_ids = batch["input_ids"][:10].to(device)
attention_mask = batch["attention_mask"][:10].to(device)
labels = batch["labels"][:10]

# Generate predictions
with torch.no_grad():
    outputs = model.generate(input_ids, attention_mask=attention_mask, max_length=64)

# Decode
predictions = tokenizer.batch_decode(outputs, skip_special_tokens=True)
ground_truth = [
    tokenizer.decode([l for l in label if l != -100], skip_special_tokens=True)
    for label in labels
]

# Display side by side with structured parsing
print("\nSample Predictions vs Ground Truth (Structured Format):")
print("="*100)
for i, (pred, truth) in enumerate(zip(predictions, ground_truth), 1):
    pred_dict = parse_structured_location(pred)
    truth_dict = parse_structured_location(truth)
    match = "✓ EXACT MATCH" if pred_dict == truth_dict else "✗ PARTIAL/NO MATCH"
    
    print(f"\n[{i}] {match}")
    print(f"Predicted:    '{pred}'")
    print(f"Ground Truth: '{truth}'")
    
    if pred_dict != truth_dict:
        # Show level-by-level comparison
        print("  Level breakdown:")
        for level in ['state', 'district', 'village', 'other_locations']:
            pred_val = pred_dict[level] or "—"
            truth_val = truth_dict[level] or "—"
            if pred_val == truth_val:
                print(f"    {level:16}: ✓ {truth_val}")
            else:
                print(f"    {level:16}: ✗ predicted='{pred_val}', actual='{truth_val}'")